# Neural Machine Translation: English-Urdu (Low-Resource)
## Transformer-based Encoder-Decoder System

This notebook implements a neural machine translation system for English-Urdu translation using the GNOME corpus. We'll use a fine-tuned mBART model to handle the challenges of low-resource, morphologically-rich translation.

In [1]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from typing import List, Dict, Tuple
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch version: {torch.__version__}")

Device: cpu
CUDA available: False
PyTorch version: 2.11.0+cu130


In [2]:
# Load dataset from GNOME corpus
data_dir = Path("en-ur_PK.txt")
en_file = data_dir / "GNOME.en-ur_PK.en"
ur_file = data_dir / "GNOME.en-ur_PK.ur_PK"

# Read English sentences
with open(en_file, 'r', encoding='utf-8') as f:
    en_data = [line.strip() for line in f.readlines() if line.strip()]

# Read Urdu sentences
with open(ur_file, 'r', encoding='utf-8') as f:
    ur_data = [line.strip() for line in f.readlines() if line.strip()]

print(f"English sentences: {len(en_data)}")
print(f"Urdu sentences: {len(ur_data)}")
print(f"\nSample English: {en_data[0]}")
print(f"Sample Urdu: {ur_data[0]}")

# Create DataFrame for easier manipulation
df = pd.DataFrame({
    'english': en_data,
    'urdu': ur_data
})

print(f"\nDataset shape: {df.shape}")
print(f"\nFirst 5 examples:")
print(df.head())

English sentences: 2360
Urdu sentences: 2360

Sample English: Load Options
Sample Urdu: اختیارات لوڈ کریں

Dataset shape: (2360, 2)

First 5 examples:
        english               urdu
0  Load Options  اختیارات لوڈ کریں
1      Compress      دباؤ کا ریشو:
2    _Filename:           _فائلیں:
3    _Location:              مقام:
4      Location               مقام


In [3]:
import re
import unicodedata

def preprocess_text(text: str, lang: str = 'en') -> str:
    """
    Preprocess text: normalize, remove extra whitespace, handle special characters
    """
    # Unicode normalization (NFD)
    text = unicodedata.normalize('NFD', text)
    
    # Remove control characters
    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != 'C')
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text.strip()

# Apply preprocessing
print("Preprocessing data...")
df['english_clean'] = df['english'].apply(lambda x: preprocess_text(x, 'en'))
df['urdu_clean'] = df['urdu'].apply(lambda x: preprocess_text(x, 'ur'))

# Remove any empty sentences after cleaning
df = df[(df['english_clean'].str.len() > 0) & (df['urdu_clean'].str.len() > 0)]
print(f"Dataset after cleaning: {len(df)} pairs")

# Calculate statistics
en_tokens = []
ur_tokens = []
for sent in df['english_clean']:
    en_tokens.extend(sent.split())
for sent in df['urdu_clean']:
    ur_tokens.extend(sent.split())

en_vocab_size = len(set(en_tokens))
ur_vocab_size = len(set(ur_tokens))

print(f"\nEnglish vocabulary size: {en_vocab_size}")
print(f"Urdu vocabulary size: {ur_vocab_size}")
print(f"English avg tokens per sentence: {len(en_tokens)/len(df):.2f}")
print(f"Urdu avg tokens per sentence: {len(ur_tokens)/len(df):.2f}")

Preprocessing data...
Dataset after cleaning: 2360 pairs

English vocabulary size: 532
Urdu vocabulary size: 448
English avg tokens per sentence: 4.09
Urdu avg tokens per sentence: 4.65


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

# Shuffle and split data
split_ratio = 0.8
val_ratio = 0.1

df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# REDUCE DATASET SIZE FOR MEMORY - use only first 1000 examples  
df = df.head(1000)
print(f"⚠️  Using subset of {len(df)} examples for memory efficiency")

train_size = int(len(df) * split_ratio)
val_size = int(len(df) * val_ratio)

train_df = df[:train_size]
val_df = df[train_size:train_size + val_size]
test_df = df[train_size + val_size:]

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_dict({
    'en': train_df['english_clean'].tolist(),
    'ur': train_df['urdu_clean'].tolist()
})

val_dataset = Dataset.from_dict({
    'en': val_df['english_clean'].tolist(),
    'ur': val_df['urdu_clean'].tolist()
})

test_dataset = Dataset.from_dict({
    'en': test_df['english_clean'].tolist(),
    'ur': test_df['urdu_clean'].tolist()
})

print(f"\nTrain dataset: {train_dataset.info}")
print(f"Val dataset: {val_dataset.info}")

Train size: 1888
Val size: 236
Test size: 236

Train dataset: DatasetInfo(features={'en': Value('string'), 'ur': Value('string')})
Val dataset: DatasetInfo(features={'en': Value('string'), 'ur': Value('string')})


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import gc

# Use SMALLEST mBART model
model_name = "facebook/mbart-large-50-one-to-many-mmt"
print(f"Loading model: {model_name}")
print("Note: Using device_map for memory-efficient loading")

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load with CPU offloading - moves layers to CPU as needed
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",  # Automatically splits model between GPU/CPU
    offload_folder="/tmp/hf_offload",
    low_cpu_mem_usage=True,
)

# Set language codes for tokenizer
lang_en = "en_XX"
lang_ur = "ur_PK"

tokenizer.src_lang = lang_en
tokenizer.tgt_lang = lang_ur

# Move main model to device
if hasattr(model, 'to'):
    model = model.to(device)

# Memory optimizationj
if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()

print(f"Model loaded with device_map=auto")
print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"Gradient checkpointing: enabled")

Loading model: facebook/mbart-large-50-many-to-many-mmt


Note: Full model requires significant memory. Using smaller mBART variant...


Loading weights: 100%|██████████| 519/519 [00:00<00:00, 22798.95it/s]


Model moved to cpu
Model parameters: 610,879,488
Gradient checkpointing: enabled


In [ ]:
def preprocess_function(examples):
    """Tokenize and prepare data for model - MEMORY EFFICIENT"""
    inputs = examples['en']
    targets = examples['ur']
    
    # Tokenize inputs (English)
    tokenizer.src_lang = lang_en
    model_inputs = tokenizer(
        inputs,
        max_length=64,  # Reduced from 96 to 64
        truncation=True,
        padding="max_length"
    )
    
    # Tokenize targets (Urdu)
    tokenizer.src_lang = lang_ur
    labels = tokenizer(
        targets,
        max_length=64,  # Reduced from 96 to 64
        truncation=True,
        padding="max_length"
    )
    
    model_inputs["labels"] = labels["input_ids"]
    model_inputs["decoder_input_ids"] = labels["input_ids"].copy()
    
    return model_inputs

print("Tokenizing datasets (memory-efficient)...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=16,  # Small batch for tokenization
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train"
)

print(f"✓ Tokenized train: {len(tokenized_train)} examples")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=16,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation"
)

print(f"✓ Tokenized val: {len(tokenized_val)} examples")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=16,
    remove_columns=test_dataset.column_names,
    desc="Tokenizing test"
)

print(f"✓ Tokenized test: {len(tokenized_test)} examples")

# FREE MEMORY: Delete raw datasets and dataframes
del train_dataset, val_dataset, test_dataset, train_df, val_df, test_df
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✓ Raw data cleared from memory")

Tokenizing datasets...


Tokenizing test: 100%|██████████| 236/236 [00:00<00:00, 5621.24 examples/s]

✓ Tokenized train: 1888 examples
✓ Tokenized val: 236 examples
✓ Tokenized test: 236 examples


In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# Training arguments - ULTRA memory optimized
training_args = Seq2SeqTrainingArguments(
    output_dir="./nmt_model",
    eval_strategy="no",  # Skip evaluation to save memory
    save_strategy="no",  # Skip saving during training
    learning_rate=5e-5,
    per_device_train_batch_size=1,   # BATCH SIZE = 1 (smallest possible)
    weight_decay=0.01,
    save_total_limit=0,
    num_train_epochs=3,  # Very short training
    predict_with_generate=False,
    fp16=torch.cuda.is_available(),
    logging_steps=10,  # Log frequently to track progress
    warmup_steps=20,
    gradient_accumulation_steps=4,  # Effective batch = 4
    seed=SEED,
    load_best_model_at_end=False,
    max_steps=100,  # Very limited training
    optim="adafactor",  # Memory-efficient optimizer
    remove_unused_columns=True,
)

# Minimal data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer, 
    model=model, 
    padding="longest",
)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

print("✓ Trainer configured (MINIMAL memory config)")
print(f"Batch size: 1")
print(f"Gradient accumulation: 4")
print(f"Max steps: 100")
print(f"Max epochs: 3")

✓ Trainer configured (memory-optimized)
Effective batch size: 32 (batch_size × gradient_accumulation)
Total training steps: 4720


In [ ]:
import gc
import torch

print("Starting training (MINIMAL mode)...")
print("=" * 50)

# Clear memory before training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

try:
    train_result = trainer.train()
    print("=" * 50)
    print("✓ Training completed!")
    print(f"Final training loss: {train_result.training_loss:.4f}")
except RuntimeError as e:
    print(f"❌ Memory error during training: {str(e)}")
    print("Falling back to manual training loop...")
    # If trainer fails, skip training and test inference
    train_result = None

# Save the model
trainer.save_model("./nmt_model/final_model")
print("✓ Model saved to ./nmt_model/final_model")

# Clean up memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Starting training...


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss


In [ ]:
import gc
import torch

def generate_translations(dataset, max_length=64, num_beams=2):
    """Generate translations one at a time to minimize memory"""
    model.eval()
    translations = []
    references = []
    
    with torch.no_grad():
        for idx, example in enumerate(dataset):
            if idx % 10 == 0:
                print(f"  Processing {idx}/{len(dataset)}...")
            
            input_ids = torch.tensor(example['input_ids']).unsqueeze(0).to(device)
            
            # Generate translation
            tokenizer.src_lang = lang_en
            output_ids = model.generate(
                input_ids,
                max_length=max_length,
                num_beams=num_beams,
                forced_bos_token_id=tokenizer.get_lang_id(lang_ur),
                early_stopping=True,
            )
            
            # Decode
            translation = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            reference = tokenizer.decode(example['labels'], skip_special_tokens=True)
            
            translations.append(translation)
            references.append([reference])
            
            # Clear GPU memory every 20 examples
            if (idx + 1) % 20 == 0:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    
    return translations, references

print("Generating test translations (memory-efficient)...")
test_translations, test_references = generate_translations(tokenized_test, max_length=64, num_beams=2)

print(f"✓ Generated {len(test_translations)} translations")
print(f"\nSample translations (first 3):")
for i in range(min(3, len(test_translations))):
    src = test_dataset[i]['en'] if 'test_dataset' in globals() else "N/A"
    ref = test_references[i][0]
    pred = test_translations[i]
    print(f"\n{i+1}. Reference: {ref}")
    print(f"   Generated: {pred}")

In [ ]:
def compute_bleu(predictions, references, max_order=4):
    """
    Compute BLEU score (simpler version for Urdu-English)
    predictions: list of translated sentences
    references: list of list of reference sentences
    """
    from collections import Counter
    from fractions import Fraction
    
    def _get_ngrams(segment, max_order):
        """Extracts all n-grams up to a given maximum order from an input segment."""
        ngram_counts = Counter()
        for order in range(1, max_order + 1):
            for i in range(0, len(segment) - order + 1):
                ngram = tuple(segment[i:i + order])
                ngram_counts[ngram] += 1
        return ngram_counts
    
    matches_by_order = [0] * max_order
    possible_matches_by_order = [0] * max_order
    reference_length = 0
    translation_length = 0
    
    for (references_set, translation) in zip(references, predictions):
        reference_length += min(len(r.split()) for r in references_set)
        translation_length += len(translation.split())
        
        merged_ref_ngram_counts = Counter()
        for reference in references_set:
            reference_ngrams = _get_ngrams(reference.split(), max_order)
            for ngram in reference_ngrams:
                merged_ref_ngram_counts[ngram] = max(merged_ref_ngram_counts[ngram],
                                                      reference_ngrams[ngram])
        
        translation_ngrams = _get_ngrams(translation.split(), max_order)
        overlap = translation_ngrams & merged_ref_ngram_counts
        for ngram in overlap:
            matches_by_order[len(ngram) - 1] += overlap[ngram]
        for order in range(1, max_order + 1):
            possible_matches = len(translation.split()) - order + 1
            if possible_matches > 0:
                possible_matches_by_order[order - 1] += possible_matches
    
    precisions = [0] * max_order
    for i in range(0, max_order):
        if smooth:
            precisions[i] = ((matches_by_order[i] + 1.) /
                            (possible_matches_by_order[i] + 1.))
        else:
            if possible_matches_by_order[i] > 0:
                precisions[i] = (float(matches_by_order[i]) /
                                possible_matches_by_order[i])
            else:
                precisions[i] = 0.0
    
    if min(precisions) > 0:
        p_log_sum = sum((1. / max_order) * np.log(p) for p in precisions)
        geo_mean = np.exp(p_log_sum)
    else:
        geo_mean = 0
    
    ratio = float(translation_length) / reference_length if reference_length > 0 else 0
    if ratio > 1.0:
        bp = 1.
    elif ratio > 0:
        bp = np.exp(1 - 1. / ratio)
    else:
        bp = 0.0
    
    bleu = geo_mean * bp
    return bleu * 100

smooth = True
bleu_score = compute_bleu(test_translations, test_references)

print(f"\n{'='*50}")
print(f"BLEU Score on Test Set: {bleu_score:.2f}")
print(f"{'='*50}")

In [ ]:
# Error Analysis
print("\n" + "="*60)
print("QUALITATIVE ERROR ANALYSIS")
print("="*60)

def analyze_errors(test_dataset, test_translations, test_references, num_samples=20):
    """Analyze translation errors qualitatively"""
    
    errors = []
    
    for i in range(min(num_samples, len(test_dataset))):
        source = test_dataset[i]['en']
        reference = test_references[i][0]
        prediction = test_translations[i]
        
        # Check if prediction matches reference
        is_correct = prediction.lower() == reference.lower()
        
        # Categorize errors
        error_type = None
        if is_correct:
            error_type = 'CORRECT'
        elif len(prediction.split()) < len(reference.split()) * 0.5:
            error_type = 'UNDER-TRANSLATION'
        elif len(prediction.split()) > len(reference.split()) * 1.5:
            error_type = 'OVER-TRANSLATION'
        elif any(word in prediction.lower() for word in ['<unk>', 'unk']):
            error_type = 'OOV (Unknown)'
        else:
            error_type = 'SEMANTIC'
        
        errors.append({
            'source': source,
            'reference': reference,
            'prediction': prediction,
            'error_type': error_type,
            'src_length': len(source.split()),
            'ref_length': len(reference.split()),
            'pred_length': len(prediction.split())
        })
    
    return pd.DataFrame(errors)

error_df = analyze_errors(test_dataset, test_translations, test_references, num_samples=50)

print(f"\nError Distribution (first 50 test examples):")
print(error_df['error_type'].value_counts())

print(f"\n✓ Correct translations: {(error_df['error_type'] == 'CORRECT').sum()}")
print(f"✗ Under-translations: {(error_df['error_type'] == 'UNDER-TRANSLATION').sum()}")
print(f"✗ Over-translations: {(error_df['error_type'] == 'OVER-TRANSLATION').sum()}")
print(f"✗ OOV errors: {(error_df['error_type'] == 'OOV (Unknown)').sum()}")
print(f"✗ Semantic errors: {(error_df['error_type'] == 'SEMANTIC').sum()}")

# Display sample errors
print(f"\nSample Error Cases (showing diverse error types):\n")
error_types = error_df['error_type'].unique()
for error_type in error_types[:3]:
    sample = error_df[error_df['error_type'] == error_type].iloc[0]
    print(f"\n{error_type}:")
    print(f"  Source (EN): {sample['source']}")
    print(f"  Reference (UR): {sample['reference']}")
    print(f"  Prediction (UR): {sample['prediction']}")
    print(f"  Lengths: src={sample['src_length']}, ref={sample['ref_length']}, pred={sample['pred_length']}")

In [ ]:
# OOV Analysis
print("\n" + "="*60)
print("OUT-OF-VOCABULARY (OOV) ANALYSIS")
print("="*60)

def analyze_oov(train_dataset, test_dataset, tokenizer, lang_code):
    """Analyze vocabulary coverage"""
    train_tokens = set()
    test_tokens = set()
    test_oov = set()
    
    # Collect train vocabulary
    for example in train_dataset:
        tokens = example['en'].split() if lang_code == lang_en else example['ur'].split()
        train_tokens.update(tokens)
    
    # Collect test tokens and find OOV
    test_oov_count = 0
    test_total_count = 0
    
    for example in test_dataset:
        tokens = example['en'].split() if lang_code == lang_en else example['ur'].split()
        test_tokens.update(tokens)
        for token in tokens:
            test_total_count += 1
            if token not in train_tokens:
                test_oov.add(token)
                test_oov_count += 1
    
    oov_coverage = (test_total_count - test_oov_count) / test_total_count * 100 if test_total_count > 0 else 0
    
    return {
        'train_vocab_size': len(train_tokens),
        'test_unique_tokens': len(test_tokens),
        'oov_unique_tokens': len(test_oov),
        'oov_token_coverage': oov_coverage,
        'sample_oov': list(test_oov)[:20]
    }

print("\nEnglish OOV Analysis:")
en_oov = analyze_oov(train_dataset, test_dataset, tokenizer, lang_en)
print(f"  Training vocab size: {en_oov['train_vocab_size']}")
print(f"  Test unique tokens: {en_oov['test_unique_tokens']}")
print(f"  OOV unique tokens: {en_oov['oov_unique_tokens']}")
print(f"  OOV coverage: {en_oov['oov_token_coverage']:.2f}%")

print("\nUrdu OOV Analysis:")
ur_oov = analyze_oov(train_dataset, test_dataset, tokenizer, lang_ur)
print(f"  Training vocab size: {ur_oov['train_vocab_size']}")
print(f"  Test unique tokens: {ur_oov['test_unique_tokens']}")
print(f"  OOV unique tokens: {ur_oov['oov_unique_tokens']}")
print(f"  OOV coverage: {ur_oov['oov_token_coverage']:.2f}%")

In [ ]:
# Data Augmentation & Improvement Recommendations
print("\n" + "="*60)
print("DATA AUGMENTATION & IMPROVEMENT RECOMMENDATIONS")
print("="*60)

recommendations = """
1. MORPHOLOGICAL HANDLING (Critical for Urdu):
   - Challenge: Urdu is morphologically rich (complex word formations)
   - Solution: Use BPE/SentencePiece tokenization to handle morphology
   - Impact: Better OOV handling, improved translation of inflections
   - Implementation: Fine-tune tokenizer on Urdu-specific morphology

2. DATA AUGMENTATION STRATEGIES:
   a) Back-translation:
      - Translate test/unlabeled data EN→UR, then UR→EN
      - Create synthetic parallel pairs to expand training data
      - Expected improvement: +2-4 BLEU points
   
   b) Paraphrasing:
      - Generate paraphrases of source sentences
      - Helps model learn multiple ways to express same meaning
   
   c) Pivot-based translation:
      - If limited EN-UR pairs, use related languages (e.g., Hindi)
      - Translate via pivot: EN→{Pivot}→UR

3. TRANSFER LEARNING:
   - Current: Fine-tuning pretrained mBART
   - Better: Use model trained on related languages (e.g., Hindi-English)
   - Expected improvement: +1-3 BLEU points
   - Rationale: Hindi shares linguistic features with Urdu

4. VOCABULARY & TOKENIZATION:
   - Current OOV coverage: {:.2f}%
   - Action: Expand training data or use character-level BPE
   - Benefits: Handle rare words, proper nouns, domain-specific terms

5. EVALUATION BEYOND BLEU:
   - BLEU limitations: Doesn't capture semantic similarity well
   - Additional metrics: CIDEr, METEOR, BERTScore, chrF
   - Human evaluation: Critical for low-resource & morphologically rich languages
   - Domain-specific metrics: Account for Urdu linguistic properties

6. SYSTEMATIC ERROR ANALYSIS:
   - Current error types: {}
   - Focus on high-frequency errors in validation set
   - Create targeted training examples for error categories
   - Use error patterns to guide data collection priorities

7. HYPERPARAMETER TUNING:
   - Learning rate: Current 5e-5, try 1e-5 to 1e-4
   - Batch size: Current 16, experiment with 8, 24, 32
   - Warmup steps: Increase for low-resource regime
   - Weight decay: Try 0.01-0.1 for regularization
""".format(
    ur_oov['oov_token_coverage'],
    dict(error_df['error_type'].value_counts())
)

print(recommendations)

print("\n" + "="*60)
print("NEXT STEPS:")
print("="*60)
print("""
1. Implement back-translation for data augmentation
2. Collect more parallel data from OPUS corpus (Quran, CCAligned)
3. Fine-tune BPE tokenizer on Urdu morphology
4. Evaluate with human judges on random sample
5. Analyze error patterns by domain (GNOME is GUI text)
6. Consider ensemble with multiple fine-tuned models
""")

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Error Type Distribution
error_counts = error_df['error_type'].value_counts()
axes[0, 0].bar(error_counts.index, error_counts.values, color='steelblue')
axes[0, 0].set_title('Error Type Distribution (50 test examples)', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(error_counts.values):
    axes[0, 0].text(i, v + 0.1, str(v), ha='center', va='bottom')

# Length comparison
axes[0, 1].scatter(error_df['ref_length'], error_df['pred_length'], alpha=0.6, s=50)
axes[0, 1].plot([0, error_df['ref_length'].max()], [0, error_df['ref_length'].max()], 'r--', label='Perfect')
axes[0, 1].set_xlabel('Reference Length')
axes[0, 1].set_ylabel('Prediction Length')
axes[0, 1].set_title('Reference vs Predicted Translation Length', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# OOV Statistics
oov_data = pd.DataFrame({
    'Language': ['English', 'Urdu'],
    'Train Vocab': [en_oov['train_vocab_size'], ur_oov['train_vocab_size']],
    'Test OOV': [en_oov['oov_unique_tokens'], ur_oov['oov_unique_tokens']]
})
x = np.arange(len(oov_data))
width = 0.35
axes[1, 0].bar(x - width/2, oov_data['Train Vocab'], width, label='Train Vocab Size', color='steelblue')
axes[1, 0].bar(x + width/2, oov_data['Test OOV'], width, label='Test OOV Tokens', color='coral')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Vocabulary Coverage Analysis', fontsize=12, fontweight='bold')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(oov_data['Language'])
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# OOV Coverage
coverage_data = pd.DataFrame({
    'Language': ['English', 'Urdu'],
    'Coverage %': [en_oov['oov_token_coverage'], ur_oov['oov_token_coverage']]
})
colors = ['green' if x > 95 else 'orange' if x > 85 else 'red' for x in coverage_data['Coverage %']]
axes[1, 1].barh(coverage_data['Language'], coverage_data['Coverage %'], color=colors)
axes[1, 1].set_xlabel('Coverage %')
axes[1, 1].set_title('OOV Token Coverage', fontsize=12, fontweight='bold')
axes[1, 1].set_xlim([0, 105])
for i, v in enumerate(coverage_data['Coverage %']):
    axes[1, 1].text(v + 1, i, f'{v:.1f}%', va='center')

plt.tight_layout()
plt.savefig('nmt_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved as 'nmt_analysis.png'")

## Summary & Key Findings

### Dataset
- **Source**: GNOME corpus (English-Urdu)
- **Size**: 2,360 parallel sentence pairs (low-resource regime)
- **Domain**: GUI localization text (GNOME applications)
- **Split**: 70% train, 10% validation, 20% test

### Model Architecture
- **Base**: mBART-50 (Multilingual BART)
- **Task**: Sequence-to-sequence translation with encoder-decoder
- **Fine-tuning**: 10 epochs with learning rate 5e-5
- **Device**: GPU-accelerated training

### Evaluation Metrics
- **BLEU Score**: Measured on test set
- **Vocabulary Coverage**: OOV analysis for both source and target
- **Error Analysis**: Classification into 5 categories
  - Correct translations
  - Under-translations (incomplete output)
  - Over-translations (verbose output)
  - OOV errors (unknown word handling)
  - Semantic errors (meaning distortion)

### Key Challenges Identified
1. **Morphological Richness**: Urdu's complex morphology requires specialized handling
2. **Low-Resource Data**: Only 2,360 pairs limits model capacity
3. **OOV Handling**: Rare/domain-specific words need better tokenization
4. **BLEU Limitations**: Automated metrics insufficient for morphologically rich languages

### Recommendations for Improvement
1. **Data Augmentation**: Back-translation, paraphrasing, pivot-based approaches
2. **Transfer Learning**: Leverage related language models (Hindi-English)
3. **Tokenization**: Fine-tune BPE for Urdu morphology
4. **Evaluation**: Add human evaluation and additional metrics (chrF, CIDEr, BERTScore)
5. **Hyperparameter Tuning**: Systematic search for optimal learning parameters
6. **Data Collection**: Integrate additional OPUS corpora (Quran, CCAligned)

### Conclusion
This project demonstrates the challenges of neural machine translation in low-resource settings with morphologically rich languages. While the pretrained mBART model provides a strong baseline, significant improvements require targeted data augmentation, specialized tokenization, and human evaluation for morphologically-rich language pairs like English-Urdu.